In [13]:
from pathlib import Path
import shutil

dataset_dir = Path("../../../Downloads/dataset_repos")

print(dataset_dir.resolve())
print(dataset_dir.exists())

/Users/senamukai/Downloads/dataset_repos
True


In [14]:
#AGENTS.mdとCLAUDE.mdがデータセットファイル内に存在する総数
agents_files = list(dataset_dir.rglob("AGENTS.md"))
claude_files = list(dataset_dir.rglob("CLAUDE.md"))

print("AGENTS:", len(agents_files))
print("CLAUDE:", len(claude_files))

AGENTS: 3703
CLAUDE: 3395


In [15]:
#AGENTS.mdとCLAUDE.mdのそれぞれに対して，https://の総数を数えるコード
def total_count_https(files):
    count = 0

    for file in files: #GitHubリポジトリのCode上を見ている
        with open(file, "r", encoding = "utf-8", errors = "ignore") as f:
            count += f.read().count("https://")
    return count

print("AGENTS:", total_count_https(agents_files))
print("CLAUDE:", total_count_https(claude_files))        


AGENTS: 4323
CLAUDE: 1833


In [16]:
"""
AGENTS.mdとCLAUDE.mdのhttps://を1つ以上含むファイルに対して，
1ファイルごとに1としてカウントし，ファイルフォルダーを作成するプログラム
"""
agents_output = Path("./https_AGENTS")
claude_output = Path("./https_CLAUDE")

agents_output.mkdir(exist_ok = True)
claude_output.mkdir(exist_ok = True)

#AGNTS.md files
agents_count = 0

for file in agents_files:

    with open(file, "r", encoding = "utf-8", errors = "ignore") as f:
        text = f.read()

    if "https://" in text:
        relative_path = file.relative_to(dataset_dir)
        save_path = agents_output / relative_path
        save_path.parent.mkdir(parents = True, exist_ok = True)
        shutil.copy2(file, save_path)
        agents_count += 1

#CLAUDE.md files
claude_count = 0

for file in claude_files:

    with open(file, "r", encoding = "utf-8", errors = "ignore") as f:
        text = f.read()
        
    if "https://" in text:
        relative_path = file.relative_to(dataset_dir)
        save_path = claude_output / relative_path
        save_path.parent.mkdir(parents = True, exist_ok = True)
        shutil.copy2(file, save_path)
        claude_count += 1

print("https:// in AGENTS.md:", agents_count)
print("https:// in CLAUDE.md:", claude_count)

https:// in AGENTS.md: 917
https:// in CLAUDE.md: 635


In [17]:
"""
RQ2:トークン消費量について取り組むためのプログラム
dataset_repos内から，AGENTS.md or CLAUDE.mdのいずれかを含むリポジトリに対し，
10Mbyte以下のリポジトリを満たすか確認し，
どちらも満たした場合，抽出しそのリポジトリ丸ごとをwork2ディレクトリ下に配置するプログラム
"""

#元々のデータセット
dataset_dir = Path("../../../Downloads/dataset_repos")
output_dir = Path("../work2/under10Mbyte_AGENTS&&CLAUDE_repos")

#10Mbyteの計算
LIMIT_SIZE = 10 * 1024 * 1024

#出力フォルダ作成
output_dir.mkdir(parents = True, exist_ok = True)

#riposのサイズを求める関数
def get_dir_size(path):
    total = 0

    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size

    return total   

#カウンタ変数の初期化
copied_repo_count = 0
skipped_no_agent_file = 0
skipped_over_10mb = 0

#riposを1個ずつ探索
for repo_dir in dataset_dir.iterdir():

    if not repo_dir.is_dir():
        continue

    #条件１: AGENTS.md or CLAUDE.md のいずれかを含むか確認
    has_agents = any(repo_dir.rglob("AGENTS.md"))
    has_claude = any(repo_dir.rglob("CLAUDE.md"))

    if not (has_agents or has_claude):
        skipped_no_agent_file += 1
        continue

    #条件2: リポジトリ全体のサイズが10Mbyte以下か確認
    repo_size = get_dir_size(repo_dir)

    if repo_size > LIMIT_SIZE:
        skipped_over_10mb += 1
        continue

    #条件1,2を満たす場合リポジトリをコピーする
    save_path = output_dir / repo_dir.name

    if save_path.exists():
        shutil.rmtree(save_path)

    shutil.copytree(
        repo_dir,
        save_path,
        symlinks = True,
        ignore_dangling_symlinks = True
    )
       
    copied_repo_count += 1

print("copied repositories:", copied_repo_count)
print("skipped no AGENTS.md or CLAUDE.md:", skipped_no_agent_file)
print("skipped over 10Mbyte:", skipped_over_10mb)


copied repositories: 3513
skipped no AGENTS.md or CLAUDE.md: 1225
skipped over 10Mbyte: 0


In [18]:
"""
dataset_repo内のリポジトリ構成がどうなっているか確認
dataset_repo内のリポジトリの中で最も容量が大きいのは何MBかを確認
コピペ(from Chat GPT)
"""

dataset_dir = Path("../../../Downloads/dataset_repos")

for i, p in enumerate(dataset_dir.iterdir()):
    print(p)
    if i == 10:
        break

#20件のリポジトリの容量を確認
for i, repo_dir in enumerate(dataset_dir.iterdir()):

    if not repo_dir.is_dir():
        continue

    size = get_dir_size(repo_dir)

    print(
        repo_dir.name,
        round(size / 1024 / 1024, 2),
        "MB"
    )

    if i == 20:
        break

#最大容量のリポジトリを求める
max_size = 0
max_repo = ""

for repo_dir in dataset_dir.iterdir():

    if not repo_dir.is_dir():
        continue

    size = get_dir_size(repo_dir)

    if size > max_size:
        max_size = size
        max_repo = repo_dir.name

print("Largest repository")
print(max_repo)
print(round(max_size / 1024 / 1024, 2), "MB")

../../../Downloads/dataset_repos/ansys§pymapdl
../../../Downloads/dataset_repos/marcobambini§gravity
../../../Downloads/dataset_repos/cloudflare§partykit
../../../Downloads/dataset_repos/nugine§s3s
../../../Downloads/dataset_repos/dotnet§maintenance-packages
../../../Downloads/dataset_repos/giselles-ai§giselle
../../../Downloads/dataset_repos/onflow§flow-go
../../../Downloads/dataset_repos/openforis§collect-earth
../../../Downloads/dataset_repos/candid82§joker
../../../Downloads/dataset_repos/ever-co§ever-teams
../../../Downloads/dataset_repos/TrueLayer§truelayer-dotnet
ansys§pymapdl 0.02 MB
marcobambini§gravity 0.0 MB
cloudflare§partykit 0.01 MB
nugine§s3s 0.0 MB
dotnet§maintenance-packages 0.0 MB
giselles-ai§giselle 0.09 MB
onflow§flow-go 0.01 MB
openforis§collect-earth 0.01 MB
candid82§joker 0.01 MB
ever-co§ever-teams 0.5 MB
TrueLayer§truelayer-dotnet 0.0 MB
tmux-python§libtmux 0.02 MB
linuxdeepin§dde-control-center 0.0 MB
greenshot§greenshot 0.02 MB
dotnet§docker-tools 0.01 MB
adaf

In [19]:
"""
RQ2:トークン消費量について取り組むためのプログラムの続き
上記フィルターをかけたものの内CLOUDE.mdを含むもののみを取り出して，
https://のリンクがあるかないかで分けるプログラム
"""

input_dir = Path("../work2/under10Mbyte_AGENTS&&CLAUDE_repos")
output_dir = Path("../work3")

https_output = output_dir / "https_CLAUDE"
no_https_output = output_dir / "no_https_CLAUDE"

https_output.mkdir(parents = True, exist_ok = True)
no_https_output.mkdir(parents = True, exist_ok = True)

#カウンタ変数の初期化
https_count = 0
no_https_count = 0
skipped_no_claude = 0

for repo_dir in input_dir.iterdir():

    if not repo_dir.is_dir():
        continue

    claude_files = list(repo_dir.rglob("CLAUDE.md"))

    #CLAUDE.mdが存在しないディレクトリは対象外
    if len(claude_files) == 0:
        skipped_no_claude += 1
        continue

    #CLAUDE.mdのファイルはhttps://のリンクを含むか
    has_https = False

    for claude_file in claude_files:
        with open(claude_file, "r", encoding = "utf-8", errors = "ignore") as f:
            text = f.read()

        if "https://" in text:
            has_https = True
            break

    if has_https:
        save_path = https_output / repo_dir.name
        https_count += 1
    else:
        save_path = no_https_output / repo_dir.name
        no_https_count += 1

    if save_path.exists():
        shutil.rmtree(save_path)

    shutil.copytree(
        repo_dir,
        save_path,
        symlinks = True,
        ignore_dangling_symlinks = True
    )

print("CLAUDErepos with https:// : ", https_count)
print("CLAUDErepos without https:// : ", no_https_count)
print("skipped no CLAUDE.md : ", skipped_no_claude)

CLAUDErepos with https:// :  547
CLAUDErepos without https:// :  1766
skipped no CLAUDE.md :  1200


In [21]:
import sys
!{sys.executable} -m pip install python-dotenv anthropic

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 931 kB 3.1 MB/s eta 0:00:01
     |████████████████████████████████| 73 kB 8.7 MB/s  eta 0:00:01
     |████████████████████████████████| 113 kB 12.3 MB/s eta 0:00:01
     |████████████████████████████████| 472 kB 25.0 MB/s eta 0:00:01
     |████████████████████████████████| 311 kB 38.7 MB/s eta 0:00:01
     |████████████████████████████████| 65 kB 18.0 MB/s eta 0:00:01
     |████████████████████████████████| 133 kB 30.3 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 32.7 MB/s eta 0:00:01
     |████████████████████████████████| 2.0 MB 7.3 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [25]:
from dotenv import load_dotenv
from anthropic import Anthropic
import os

load_dotenv()

API_KEY = os.getenv("ANTHROPIC_API_KEY")
client = Anthropic(api_key = API_KEY)
if API_KEY is None:
    raise ValueError("APIキーを読み込めませんでした")

print("APIキーの読み込みに成功")

APIキーの読み込みに成功


In [26]:
PROMPT = """
You are an AI coding agent.
Read the following CLAUDE.md carefully.
Summarize the repository-specific development instructions that you must follow.

<CLAUDE.md>
{claude_content}
</CLAUDE.md>
"""

In [29]:
#1つのCLAUDE.mdファイルを用いてAnthropic APIが動くかテストするプログラム
claude_file = Path("../work3/https_CLAUDE/0xJacky§nginx-ui/CLAUDE.md")

with open(claude_file, "r", encoding = "utf-8", errors = "ignore") as f:
    text = f.read()

response = client.messages.create(
    model = "claude-sonnet-4-5",
    max_tokens = 500,
    messages = [{
                    "role" : "user",
                    "content" : PROMPT.format(
                        claude_content = text
                    )
        }   
    ]
)

print("Input tokens :", response.usage.input_tokens)

Input tokens : 1106


In [30]:
import sys
!{sys.executable} -m pip install requests beautifulsoup4 pandas matplotlib

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 64 kB 1.4 MB/s  eta 0:00:01
     |████████████████████████████████| 109 kB 2.8 MB/s eta 0:00:01
     |████████████████████████████████| 10.8 MB 2.7 MB/s eta 0:00:01    |████████████████                | 5.4 MB 2.4 MB/s eta 0:00:03
     |████████████████████████████████| 7.8 MB 73.1 MB/s eta 0:00:01
     |████████████████████████████████| 131 kB 6.1 MB/s eta 0:00:01
     |████████████████████████████████| 299 kB 68.3 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 37.5 MB/s eta 0:00:01
     |████████████████████████████████| 349 kB 3.8 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 71.7 MB/s eta 0:00:01
     |████████████████████████████████| 249 kB 13.3 MB/s eta 0:00:01
     |████████████████████████████████| 2.9 MB 24.1 MB/s eta 0:00:01
     |████████████████████████████████| 122 kB 6.8 MB/s eta 0:00:01
     |███████████████████████████

In [33]:
import re
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

In [35]:
"""
RQ2コード
"""
MODEL = "claude-sonnet-4-5"

https_dir = Path("../work3/https_CLAUDE")
no_https_dir = Path("../work3/no_https_CLAUDE")

output_csv = Path("../work4/claude_token_counts_with_link_content.csv")

PROMPT = """
You are an AI coding agent.
Read the following CLAUDE.md and linked external documents carefully.
Understand the repository-specific development instructions, coding conventions,
build/test commands, and important constraints.

Do not modify any files.
Return a concise summary of what you must follow when working in this repository.

<CLAUDE_MD_AND_LINKED_CONTEXT>
{context}
</CLAUDE_MD_AND_LINKED_CONTEXT>
"""

#巨大ページ対策
MAX_LINKS_PER_REPO = 10
MAX_CHARS_PER_LINK = 30000
REQUEST_TIMEOUT = 10

def extract_https_urls(text):
    urls = re.findall(r'https://[^\s\)\]\}"\'<>]+', text)

    cleaned = []
    for url in urls:
        url = url.rstrip(".,;:")
        cleaned.append(url)

    #重複削除
    return list(dict.fromkeys(cleaned))

def fetch_url_text(url):
    try:
        res = requests.get(
            url,
            timeout = REQUEST_TIMEOUT,
            headers = {
                "User-Agent": "Mozilla/5.0 token-count-research"
            }
        )

        if res.status_code != 200:
            return ""

        content_type = res.headers.get("Content-Type", "")

        # HTMLなら本文を抽出する
        if "text/html" in content_type:
            soup = BeautifulSoup(res.text, "html.parser")

            for tag in soup(["script", "style", "nav", "footer", "header"]):
                tag.decompose()

            text = soup.get_text(separator = "\n")
        else:
            text = res.text

        text = "\n".join(line.strip() for line in text.splitlines() if line.strip())

        return text[:MAX_CHARS_PER_LINK]

    except Exception:
        return ""


def build_context_from_claude(repo_dir, include_link_content):
    claude_files = list(repo_dir.rglob("CLAUDE.md"))

    claude_texts = []
    all_urls = []

    for file in claude_files:
        text = file.read_text(encoding = "utf-8", errors = "ignore")
        relative_path = file.relative_to(repo_dir)

        claude_texts.append(
            f"\n\n--- FILE: {relative_path} ---\n{text}"
        )

        all_urls.extend(extract_https_urls(text))

    all_urls = list(dict.fromkeys(all_urls))

    context = "\n".join(claude_texts)

    fetched_count = 0
    failed_count = 0

    if include_link_content:
        for url in all_urls[:MAX_LINKS_PER_REPO]:
            link_text = fetch_url_text(url)

            if link_text:
                context += f"\n\n--- LINK CONTENT: {url} ---\n{link_text}"
                fetched_count += 1
            else:
                failed_count += 1

            time.sleep(0.5)

    return context, len(claude_files), all_urls, fetched_count, failed_count


def count_input_tokens(context):
    prompt_text = PROMPT.format(context = context)

    result = client.messages.count_tokens(
        model = MODEL,
        messages = [
            {
                "role" : "user",
                "content" : prompt_text
            }
        ]
    )

    return result.input_tokens


rows = []


def process_group(base_dir, group_name, include_link_content):
    repo_dirs = [p for p in base_dir.iterdir() if p.is_dir()]

    for i, repo_dir in enumerate(repo_dirs, start=1):
        try:
            context, claude_file_count, urls, fetched_count, failed_count = build_context_from_claude(
                repo_dir,
                include_link_content = include_link_content
            )

            if not context.strip():
                continue

            input_tokens = count_input_tokens(context)

            rows.append({
                "group" : group_name,
                "repository" : repo_dir.name,
                "claude_file_count" : claude_file_count,
                "https_count" : len(urls),
                "fetched_link_count" : fetched_count,
                "failed_link_count" : failed_count,
                "chars" : len(context),
                "lines" : len(context.splitlines()),
                "input_tokens" : input_tokens
            })

            print(f"[{group_name}] {i}/{len(repo_dirs)} {repo_dir.name}: {input_tokens}")

            time.sleep(0.3)

        except Exception as e:
            print("ERROR:", group_name, repo_dir.name, e)


# httpsあり群(https_CLAUDE)：CLAUDE.md + リンク先本文のトークンカウント
process_group(https_dir, "https_with_link_content", include_link_content = True)

# httpsなし群(no_https_CLAUDE)：CLAUDE.mdのみのトークンカウント
process_group(no_https_dir, "no_https", include_link_content = False)

df = pd.DataFrame(rows)
df.to_csv(output_csv, index=False)

df.head()

[https_with_link_content] 1/547 ever-co§ever-teams: 14125
[https_with_link_content] 2/547 maxmind§minfraud-api-node: 8958
[https_with_link_content] 3/547 glific§glific-frontend: 1846


KeyboardInterrupt: 

In [ ]:
#集計してグラフを描画
df.groupby("group")["input_tokens"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)

summary = df.groupby("group")["input_tokens"].mean()

summary.plot(kind = "bar")

plt.title("Average input tokens: CLAUDE.md with linked content")
plt.xlabel("Group")
plt.ylabel("Input tokens")
plt.xticks(rotation = 0)
plt.show()